# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('east')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[0 3]'; Hunter 2 is at '[1 3]'; Monster is at '[1 1]'

...1.
.M.2.
.....
.....
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'teleport'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[0 2]'; Hunter 2 is at '[3 2]'; Monster is at '[1 1]'

..1..
.M...
.....
..2..
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'north'
Monster chose action 'west'

=====

Step 2:
Hunter 1 is at '[0 1]'; Hunter 2 is at '[2 2]'; Monster is at '[1 0]'

.1...
M....
..2..
.....
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'east'
Monster chose action 'stay'

=====

Step 3:
Hunter 1 is at '[0 0]'; Hunter 2 is at '[2 3]'; Monster is at '[1 0]'

1....
M....
...2.
.....
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'east'
Monster chose action 'stay'

=====

Step 4:
Hunter 1 is at '[0 0]'; Hunter 2 is at '[2 4]'; Monster is at '[1 0]'

1....
M....
....2
.....
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'south'
Monster chose 

('M', nan)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [7]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('M', nan), ('M', nan), ('M', nan), ('1', 21), ('2', 17), ('1', 68), ('1', 48), ('M', nan), ('M', nan), ('2', 34), ('1', 17), ('M', nan), ('M', nan), ('2', 70), ('1', 2), ('M', nan), ('M', nan), ('2', 79), ('2', 51), ('2', 4), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 96), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 58), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 38), ('M', nan), ('M', nan), ('2', 86), ('1', 94), ('2', 28), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 31), ('M', nan), ('1', 83), ('M', nan), ('M', nan), ('2', 76), ('2', 25), ('2', 91), ('M', nan), ('1', 84), ('1', 9), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 56), ('2', 54), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 64), ('M', nan), ('M', nan), ('2', 28), ('M', nan), ('2', 5), ('1', 12), ('M', nan), ('2', 28), ('M', nan), ('M', nan), ('M', nan), ('

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [8]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 16
Hunter 2 won: 18
Monster 1 won: 66


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [9]:
# here goes your agent implementation
import numpy as np

# Chiến lược 1: Greedy Reflex Agent (Đuổi theo khoảng cách Manhattan)
def greedy_hunter_agent_function(hunter_location, monster_location):
    row_diff = monster_location[0] - hunter_location[0]
    col_diff = monster_location[1] - hunter_location[1]
    
    # Đã đến vị trí quái vật
    if row_diff == 0 and col_diff == 0:
        return 'north'
    
    # Ưu tiên giảm trục có khoảng cách lớn hơn
    if abs(row_diff) >= abs(col_diff):
        return 'south' if row_diff > 0 else 'north'
    else:
        return 'east' if col_diff > 0 else 'west'

# Chiến lược 2: Smart Teleport Agent (Teleport khi ở xa, đuổi khi ở gần)
def smart_teleport_hunter_agent_function(hunter_location, monster_location):
    row_diff = monster_location[0] - hunter_location[0]
    col_diff = monster_location[1] - hunter_location[1]
    manhattan_dist = abs(row_diff) + abs(col_diff)
    
    # Nếu khoảng cách quá xa (> 12 ô), kích hoạt teleport để ngẫu nhiên nhảy lại gần
    if manhattan_dist > 12:
        return 'teleport'
    
    # Khi đã ở gần, di chuyển chính xác tới quái vật
    if abs(row_diff) >= abs(col_diff):
        return 'south' if row_diff > 0 else 'north'
    else:
        return 'east' if col_diff > 0 else 'west'

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [10]:
# Your experimentation code goes here
def run_experiment(h1_func, h2_func, h1_name, h2_name, n_size=20, max_steps=100, runs=100):
    results = [arena_environment(h1_func, h2_func, monster_agent_function_simple, n=n_size, max_steps=max_steps, visualize=False) for _ in range(runs)]
    results = np.array(results)
    
    h1_wins = np.sum(results[:, 0] == "1")
    h2_wins = np.sum(results[:, 0] == "2")
    m_wins = np.sum(results[:, 0] == "M")
    
    print(f"=== Thử nghiệm Đấu trường {n_size}x{n_size} ({runs} lượt) ===")
    print(f"- {h1_name} thắng: {h1_wins} lần ({(h1_wins/runs)*100:.1f}%)")
    print(f"- {h2_name} thắng: {h2_wins} lần ({(h2_wins/runs)*100:.1f}%)")
    print(f"- Monster thắng: {m_wins} lần ({(m_wins/runs)*100:.1f}%)\n")

# Thử nghiệm 1: Greedy Hunter VS Random Hunter (Đấu trường 20x20)
run_experiment(greedy_hunter_agent_function, simple_randomized_hunter_agent_function, 
               "Greedy Hunter", "Random Hunter", n_size=20, max_steps=100)

# Thử nghiệm 2: Greedy Hunter VS Smart Teleport Hunter (Đấu trường 30x30)
run_experiment(greedy_hunter_agent_function, smart_teleport_hunter_agent_function, 
               "Greedy Hunter", "Smart Teleport Hunter", n_size=30, max_steps=150)

=== Thử nghiệm Đấu trường 20x20 (100 lượt) ===
- Greedy Hunter thắng: 96 lần (96.0%)
- Random Hunter thắng: 4 lần (4.0%)
- Monster thắng: 0 lần (0.0%)

=== Thử nghiệm Đấu trường 30x30 (100 lượt) ===
- Greedy Hunter thắng: 26 lần (26.0%)
- Smart Teleport Hunter thắng: 74 lần (74.0%)
- Monster thắng: 0 lần (0.0%)



## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

**Phân tích chiến lược & Thử nghiệm:**

1. **Chiến lược chọn hành động:**
   * **Greedy Hunter:** Luôn giảm khoảng cách Manhattan bằng cách đi trực diện về phía quái vật. Chiến lược này cho tỷ lệ thắng áp đảo hoàn toàn (>90%) khi thi đấu với Random Hunter vì quỹ đạo di chuyển có mục tiêu rõ ràng.
   * **Smart Teleport Hunter:** Khi ở đấu trường lớn ($30 \times 30$), nếu khoảng cách quá xa ($>12$ ô), dịch chuyển tức thời (`teleport`) giúp rút ngắn quãng đường tiếp cận ban đầu ngẫu nhiên trước khi chuyển sang áp sát.

2. **Ảnh hưởng của kích thước Arena ($20 \times 20$ vs $30 \times 30$):**
   * Trong đấu trường lớn hơn ($30 \times 30$), quãng đường di chuyển trung bình tăng lên rõ rệt. Khi khoảng cách ban đầu xa, tác tử `Smart Teleport` có lợi thế vượt qua các mốc khoảng cách dài nhanh hơn so với việc chỉ đi bộ từ từ.

3. **Việc sử dụng Teleportation:**
   * Không nên dùng `teleport` liên tục vì đích đến là ngẫu nhiên. Chỉ nên dùng khi khoảng cách tới quái vật quá lớn để tìm cơ hội "nhảy" tới vị trí gần hơn.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

**1. Cải tiến cho Quái vật (Monster Agent):**
* Hiện tại quái vật chỉ di chuyển ngẫu nhiên và đứng yên 50% thời gian.
* Để né tránh tốt hơn, quái vật có thể tính toán khoảng cách tới **cả hai Thợ săn** (`hunter_1_pos` và `hunter_2_pos`) và chọn hướng di chuyển sao cho **tổng khoảng cách (hoặc khoảng cách tới thợ săn gần nhất) là lớn nhất**.

**2. Chuyển đổi sang Môi trường Hợp tác (Cooperative Environment):**
* **Chia sẻ thông tin & Phân công khu vực:** Hai thợ săn có thể chia sẻ vị trí của nhau. Thợ săn 1 có thể dồn quái vật về một góc, trong khi Thợ săn 2 di chuyển chặn đầu hướng thoát của quái vật.
* **Giao tiếp/Tín hiệu (Communication):** Thêm cơ chế cho phép thợ săn phát tín hiệu/trạng thái (ví dụ: "Đang tiếp cận từ hướng Bắc", "Chặn phía Nam") giúp cả hai phối hợp bắt quái vật với tổng số bước ít nhất cho cả đội.

# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.